In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.0 --extra-index-url https://enhantica.github.io/edi-download/pypi/simple/


# Rietveld fit: LiNiO₂, ECHIDNA

Fit the layered oxide LiNiO₂ to constant-wavelength neutron powder data from ECHIDNA at ANSTO.
Li and Ni mix between two sites, held to full occupancy by constraints, and the profile has
Finger-Cox-Jephcoat asymmetry.

The project is the catalogue case `pd-neut-cwl_lmo-echidna`, the one the command line and the
app's Examples open. Li and Ni share their sites through five relations (displacements and
occupancies), kept from the start.

<span style="color:#398648;background:#39864821;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Fitting</span> <span style="color:#96601d;background:#96601d21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Single</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">pd</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">neut</span> <span style="color:#424242;background:#42424221;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">cwl</span> <span style="color:#0084c7;background:#0084c721;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Bragg</span> <span style="color:#00796b;background:#00796b21;border-radius:3px;padding:0.1em 0.4em;font-size:0.85em">Crystal structure</span>

In [2]:
import easydiffraction as edi

## Load the project

In [3]:
project_path = edi.download_data('pd-neut-cwl_lmo-echidna', destination='data')
project = edi.Project.load(project_path)
structure = project.structures['lmo']
experiment = project.experiments['echidna']
project.display.constraints()

Data 'pd-neut-cwl_lmo-echidna' is at data/pd-neut-cwl_lmo-echidna-134fba941254/pd-neut-cwl_lmo-echidna


View(html=<78,121 characters>, height=460)

The project marks what its fit frees. The fit below frees it in stages, so start with all of it
fixed.

In [4]:
for parameter in project.free_parameters:
    parameter.free = False

In [5]:
project.display.pattern('echidna')

View(html=<450,384 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scale, 2θ zero and cell

The scale, the zero shift and the two cell lengths put the peaks in place and at the right
height.

In [6]:
experiment.linked_structures['lmo'].scale.free = True
experiment.instrument.calib_twotheta_offset.free = True
structure.cell.length_a.free = True
structure.cell.length_c.free = True

In [7]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,lmo,cell,,length_a,2.88,,0,30,Å
2,lmo,cell,,length_c,14.18,,0,30,Å
3,echidna,instrument,,calib_twotheta_offset,0,,-inf,inf,deg
4,echidna,linked_structure,lmo,scale,10,,-inf,inf,


In [8]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       29.38        81.99
      1      0.003     11.32        12.18   85.1% ↓
      2      0.006      8.05         6.16   49.4% ↓
      3      0.009      8.05         6.16   0.1% ↓
      4      0.013      8.05         6.16   
  done · converged · 4 iters · reduced χ² 6.16 · Rwp 8.05 % · 0.02 s


View(html=<81,914 characters>, height=376)

In [9]:
project.display.pattern('echidna')

View(html=<451,853 characters>, height=480)

### Sample displacement and transparency

The sample sits a little off the diffractometer axis and the beam goes into it; both shift the
peaks with angle, unlike the zero.

In [10]:
experiment.instrument.calib_sample_displacement.free = True
experiment.instrument.calib_sample_transparency.free = True

In [11]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 6 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.05         6.16
      1      0.003      8.04         6.14   0.3% ↓
      2      0.005      8.04         6.14   
      3      0.009      8.04         6.14   
  done · converged · 3 iters · reduced χ² 6.14 · Rwp 8.04 % · 0.01 s


View(html=<84,251 characters>, height=376)

In [12]:
project.display.pattern('echidna')

View(html=<452,027 characters>, height=480)

### Background

The nine background points follow the level between the peaks.

In [13]:
for point in experiment.background:
    point.intensity.free = True

In [14]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 15 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        8.04         6.16
      1      0.005      6.46         3.98   35.4% ↓
      2      0.008      6.46         3.98   
      3      0.012      6.46         3.98   
  done · converged · 3 iters · reduced χ² 3.98 · Rwp 6.46 % · 0.02 s


View(html=<93,148 characters>, height=376)

In [15]:
project.display.pattern('echidna')

View(html=<480,790 characters>, height=480)

### Peak widths

In [16]:
experiment.peak.broad_gauss_u.free = True
experiment.peak.broad_gauss_v.free = True
experiment.peak.broad_gauss_w.free = True
experiment.peak.broad_lorentz_y.free = True

In [17]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 19 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.46         3.99
      1      0.004      6.39         3.90   2.2% ↓
      2      0.006      6.39         3.90   
      3      0.010      6.39         3.90   
  done · converged · 3 iters · reduced χ² 3.90 · Rwp 6.39 % · 0.01 s


View(html=<97,432 characters>, height=376)

In [18]:
project.display.pattern('echidna')

View(html=<480,712 characters>, height=480)

### Oxygen position

O's height along c is the only coordinate the symmetry leaves free.

In [19]:
structure.atom_sites['O'].fract_z.free = True

In [20]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.39         3.90
      1      0.005      6.20         3.67   5.9% ↓
      2      0.008      6.20         3.67   
      3      0.012      6.20         3.67   
  done · converged · 3 iters · reduced χ² 3.67 · Rwp 6.20 % · 0.02 s


View(html=<98,502 characters>, height=376)

In [21]:
project.display.pattern('echidna')

View(html=<480,639 characters>, height=480)

### Displacements

Li1's, Ni2's and O's; Ni1 and Li2 follow by the relations.

In [22]:
structure.atom_sites['Li1'].adp_iso.free = True
structure.atom_sites['Ni2'].adp_iso.free = True
structure.atom_sites['O'].adp_iso.free = True

In [23]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        6.20         3.67
      1      0.004      5.36         2.75   25.0% ↓
      2      0.007      5.31         2.70   2.0% ↓
      3      0.010      5.31         2.70   0.1% ↓
      4      0.015      5.31         2.70   
  done · converged · 4 iters · reduced χ² 2.70 · Rwp 5.31 % · 0.02 s


View(html=<101,646 characters>, height=376)

In [24]:
project.display.pattern('echidna')

View(html=<480,674 characters>, height=480)

### Li/Ni mixing

Last, Li1's occupancy, which sets how much Ni sits on the Li site; it moves with the
displacements, so it comes after them.

In [25]:
structure.atom_sites['Li1'].occupancy.free = True

In [26]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-cwl_lmo-echidna, 3002 pts fitted / 3200 loaded, 24 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                        5.31         2.70
      1      0.005      5.01         2.40   11.1% ↓
      2      0.008      4.99         2.39   0.5% ↓
      3      0.011      4.99         2.39   
      4      0.016      4.99         2.39   
  done · converged · 4 iters · reduced χ² 2.39 · Rwp 4.99 % · 0.02 s


View(html=<102,708 characters>, height=376)

In [27]:
project.display.pattern('echidna')

View(html=<480,619 characters>, height=480)

In [28]:
project.display.fit.correlations()

,1,2,3,4
1 lmo.cell.length_a,,,,
2 echidna.instrument.calib_twotheta_offset,0.89,,,
3 echidna.instrument.calib_sample_displacement,-0.94,-0.90,,
4 echidna.instrument.calib_sample_transparency,-0.74,-0.95,0.75,


In [29]:
project.display.structure('lmo')

View(html=<4,415,467 characters>, height=660)

## Save the project

In [30]:
project.save_as('pd-neut-cwl_lmo-echidna')
project.report.save_html('pd-neut-cwl_lmo-echidna.html')

Saving project to 'pd-neut-cwl_lmo-echidna'
Saving report to 'pd-neut-cwl_lmo-echidna.html'
